## Chat Bot Evaluation

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")

os.environ["LANGCHAIN_TRACING"] = "true"


In [ ]:
## Create the datapoints
from langsmith import Client
client = Client()

## define the dataset - these are your test data
dataset_name = "Simple Kaushal Chatbots Evaluation"

# Only create the dataset once - re-running would raise a conflict error
if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(dataset_name)
    client.create_examples(
        dataset_id=dataset.id,
        examples=[
            {
                "inputs": {"question": "What is LangChain?"},
                "outputs": {"answer": "A framework for building LLM applications"},
            },
            {
                "inputs": {"question": "What is LangSmith?"},
                "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
            },
            {
                "inputs": {"question": "What is OpenAI?"},
                "outputs": {"answer": "A company that creates Large Language Models"},
            },
            {
                "inputs": {"question": "What is Google?"},
                "outputs": {"answer": "A technology company known for search"},
            },
            {
                "inputs": {"question": "What is Mistral?"},
                "outputs": {"answer": "A company that creates Large Language Models"},
            }
        ]
    )

## Define Metrics (LLM AS A Judge)

In [22]:
import anthropic
from langsmith import wrappers

# Initialize the Anthropic client
anthropic_client = wrappers.wrap_anthropic(
    anthropic.Anthropic()
)

eval_instructions = "You are an expert professor grading student answers."


# Your correctness evaluator (LLM as a judge)
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    # The target function failed for this example - nothing to grade
    if "response" not in outputs:
        return False

    user_content = f"""
Question:
{inputs['question']}

Reference answer:
{reference_outputs['answer']}

Predicted answer:
{outputs['response']}

Judge whether the predicted answer is semantically correct.
Reply with exactly CORRECT or INCORRECT.
"""

    result = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=20,
        system=eval_instructions,
        messages=[
            {"role": "user", "content": user_content}
        ]
    )

    return result.content[0].text.strip().upper() == "CORRECT"

In [23]:
## Concision - checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    if "response" not in outputs:
        return False
    return len(outputs["response"]) < 2 * len(reference_outputs["answer"])


## Run Evaluations

In [24]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

# Your application function
def my_app(question: str, instructions: str = default_instructions) -> str:
    response = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system=instructions,
        messages=[
            {
                "role": "user",
                "content": question
            }
        ]
    )

    return response.content[0].text

In [25]:
### Call my_app for every datapoints
def ls_target(inputs: dict) -> dict:
    return {
        "response": my_app(inputs["question"])
    }

In [26]:

## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="claude-sonnet-4-6-chatbot"
)

View the evaluation results for experiment: 'claude-sonnet-4-6-chatbot-959087d3' at:
https://smith.langchain.com/o/62eb9a75-1b2a-45f3-b5f3-3c626cd91ac1/datasets/e8c6db1d-bb04-418c-ab97-748dcb243ce9/compare?selectedSessions=63972bab-591b-4ba1-bca3-63a565b8d23f




5it [00:14,  2.99s/it]


### Evaluation For RAG

In [29]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [doc for sublist in docs for doc in sublist]

print(f"Loaded {len(docs_list)} documents")

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,
    chunk_overlap=0,
)

doc_splits = text_splitter.split_documents(docs_list)
print(f"Created {len(doc_splits)} chunks")

# Initialize local embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Create vector store
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embeddings,
)

# Create retriever (retrieve top 6 chunks)
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 6}
)

Loaded 3 documents
Created 187 chunks


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2556.47it/s]


In [30]:

retriever.invoke("what is agents")

[Document(id='639832eb-67e1-4eee-8e37-2400d172823b', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [32]:

from langchain.chat_models import init_chat_model
llm=init_chat_model("claude-sonnet-4-6")
llm

ChatAnthropic(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2', 'langchain-anthropic': '1.7.4'}}, profile={'name': 'Claude Sonnet 4.6', 'release_date': '2026-02-17', 'last_updated': '2026-03-13', 'open_weights': False, 'max_input_tokens': 1000000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['low', 'medium', 'high', 'max'], 'reasoning_effort_default': 'high'}, model='claude-sonnet-4-6', max_tokens=128000, anthropic_api_url='https://api.anthropic.com', anthropic_api_key=SecretStr('**********'), model_kwargs={})

In [33]:
from langsmith import traceable

## Add decorator
@traceable()
def rag_bot(question:str)->dict:
    ## Relevant context
    docs=retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""
    
    ## llm invoke

    ai_msg=llm.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer":ai_msg.content,"documents":docs}

In [35]:
response=rag_bot("what is agents")
response

{'answer': 'An agent in the context of LLM-powered systems is an autonomous entity that uses a large language model (LLM) as its core controller or "brain." It is complemented by key components such as planning, memory, and tool use to handle complex tasks. Examples of such agents include AutoGPT, GPT-Engineer, and BabyAGI.',
 'documents': [Document(id='639832eb-67e1-4eee-8e37-2400d172823b', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, compleme

In [31]:
from langsmith import Client

client=Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

### create the daatset and example in LAngsmith
dataset_name="RAG Test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['0f39229c-bd0a-49b0-b20f-1ca4eae60ef9',
  'c2a29a4f-8913-4b3a-85c3-a1af3008b4f4',
  '848cb53b-a976-4fb5-88b6-6b9a4ff9ccc3'],
 'count': 3,
 'as_of': '2026-10-24T18:29:47.049970915Z'}

### Evaluators or Metrics
Correctness: Response vs reference answer
Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
Mode: Requires a ground truth (reference) answer supplied through a dataset
Evaluator: Use LLM-as-judge to assess answer correctness.

In [36]:
from typing_extensions import Annotated,TypedDict

## Correctness Output Schema

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

## correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

from langchain_anthropic import ChatAnthropic

grader_llm=ChatAnthropic(model="claude-sonnet-4-6",temperature=0).with_structured_output(CorrectnessGrade,
                                                                         method="json_schema",strict=True)
## evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])
    return grade["correct"]


### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [37]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0).with_structured_output(RelevanceGrade, method="json_schema", strict=True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [38]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0).with_structured_output(GroundedGrade, method="json_schema", strict=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

### Retrieval Relevance: Retrieved docs vs input

In [39]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### Run the evaluation

In [40]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, claude-sonnet-4-6-preview"},
)
# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-f26e9f8a' at:
https://smith.langchain.com/o/62eb9a75-1b2a-45f3-b5f3-3c626cd91ac1/datasets/0219dfbe-0979-436a-8e52-b0b6a1cbd223/compare?selectedSessions=3abb4314-59a2-4438-bdcd-3af06f8759c4




3it [01:08, 22.80s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What are the types of biases that can arise wi...,"According to Zhao et al. (2021), three types o...",[page_content='Text: i'll bet the video game i...,None,The biases that can arise with few-shot prompt...,True,True,True,True,4.503885,0f39229c-bd0a-49b0-b20f-1ca4eae60ef9,01a16f77-8365-7902-9566-9723a5b9438d
1,How does the ReAct agent use self-reflection?,"Based on the documents, ReAct itself does not ...",[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",True,True,True,True,4.173511,848cb53b-a976-4fb5-88b6-6b9a4ff9ccc3,01a16f77-d9cb-7ea0-8de6-646b1611e155
2,What are five types of adversarial attacks?,"Based on the document, the five types of adver...",[page_content='Adversarial attacks are inputs ...,None,Five types of adversarial attacks are (1) Toke...,True,True,True,True,2.916661,c2a29a4f-8913-4b3a-85c3-a1af3008b4f4,01a16f78-37a5-79f0-ad36-a63c6f21c0a8
